# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_01 — Data Vendor Interface and Schema Validation

### Research question

How can a quantitative finance research pipeline fetch market data from different vendors while enforcing a single canonical schema that downstream models can trust?

This notebook builds the first infrastructure layer of the repository:

1. define a vendor-neutral market data request;
2. define a canonical OHLCV schema;
3. implement a synthetic offline data vendor;
4. sketch an optional Yahoo Finance / `yfinance` adapter;
5. validate schema, dtypes, duplicates, timestamps, price logic, and metadata;
6. generate machine-readable validation reports;
7. save validated data with a manifest and dataset fingerprint;
8. discuss modern data-contract and data-quality practices.

The goal is to prevent a common quant research failure mode:

> A model appears to work, but only because inconsistent data formats, silent missing values, duplicate timestamps, adjusted/unadjusted price confusion, or vendor-specific quirks leaked into the pipeline.

## 1. Why data interfaces matter

A research notebook can often get away with:

```python
df = yf.download("SPY")
```

A research platform cannot.

As soon as the repository has multiple models and multiple data sources, we need a stable contract:

- every vendor adapter should return the same column names;
- timestamps should use a consistent timezone convention;
- adjusted and unadjusted prices should not be silently mixed;
- duplicates should be detected;
- missing values should be visible;
- validation failures should be reported before modelling begins.

This notebook therefore separates two ideas:

| Layer | Purpose |
|---|---|
| **Vendor adapter** | Fetches raw data from one source |
| **Canonical schema** | Standardises the format used by the rest of the repo |
| **Validator** | Checks whether the data is safe enough to consume |
| **Manifest** | Records what was fetched, when, from where, and under which schema |

This is not glamorous, but it is foundational. Every later notebook depends on this layer.

## 2. Canonical OHLCV schema v1

For this repository, the canonical daily OHLCV schema is:

| Column | Type | Meaning |
|---|---|---|
| `symbol` | string | Asset identifier |
| `timestamp` | datetime64 UTC | Observation timestamp |
| `open` | float64 | Opening price |
| `high` | float64 | Highest price over the bar |
| `low` | float64 | Lowest price over the bar |
| `close` | float64 | Closing price |
| `adj_close` | float64 | Adjusted closing price, if available |
| `volume` | float64 | Traded volume |
| `currency` | string | Quoted currency |
| `source` | string | Vendor/source identifier |
| `schema_version` | string | Data contract version |
| `downloaded_at` | datetime64 UTC | Time the data was generated or fetched |

The composite key is:

$$
(\text{symbol}, \text{timestamp})
$$

There should be no duplicate rows under this key.

A downstream model should be able to assume:

1. prices are finite and positive;
2. volume is finite and non-negative;
3. `high >= low`;
4. `high >= open, close`;
5. `low <= open, close`;
6. timestamps are timezone-aware and normalised to UTC;
7. rows are sorted by `symbol` and `timestamp`.

## 3. Imports and configuration

This notebook only requires standard scientific Python libraries.

The optional `YFinanceVendor` adapter later in the notebook requires `yfinance`, but the main workflow uses a synthetic vendor so that the notebook is fully reproducible offline.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Protocol
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

## 4. Data request object

The request object makes vendor calls explicit.

Instead of passing loose arguments around the codebase, every vendor receives the same structured request.

In [ ]:
@dataclass(frozen=True)
class VendorRequest:
    """
    Vendor-neutral market data request.

    Parameters
    ----------
    symbols:
        List of symbols to fetch.

    start:
        Start date as a string interpretable by pandas.

    end:
        End date as a string interpretable by pandas.

    interval:
        Bar interval. This notebook focuses on daily bars.

    currency:
        Currency label to attach to the canonical data.

    auto_adjust:
        Whether a vendor should return adjusted prices where available.
        The exact meaning depends on the vendor.
    """
    symbols: list[str]
    start: str
    end: str
    interval: str = "1d"
    currency: str = "USD"
    auto_adjust: bool = False


request = VendorRequest(
    symbols=["AAA", "BBB", "CCC", "DDD"],
    start="2022-01-01",
    end="2024-12-31",
    interval="1d",
    currency="USD",
    auto_adjust=False
)

request

## 5. Vendor interface

Each vendor adapter should implement one method:

```python
fetch_ohlcv(request: VendorRequest) -> pd.DataFrame
```

The adapter may fetch from:

- a public API;
- a local file;
- a database;
- a synthetic generator;
- a paid vendor;
- a cached research lake.

The important rule is that after standardisation, the returned DataFrame must satisfy the canonical schema.

In [ ]:
class MarketDataVendor(Protocol):
    """
    Protocol for market data vendors.
    """
    name: str

    def fetch_ohlcv(self, request: VendorRequest) -> pd.DataFrame:
        """
        Fetch OHLCV data and return a DataFrame.
        """
        ...

## 6. Synthetic offline vendor

The synthetic vendor exists for reproducibility.

It generates OHLCV data that behaves roughly like equity prices, but without relying on internet access or external APIs.

The process is not intended to be a realistic market model. It is only designed to produce data with the right shape and enough noise to test the infrastructure.

In [ ]:
class SyntheticOHLCVVendor:
    """
    Synthetic vendor producing reproducible OHLCV data.

    This is useful for testing the data pipeline without relying on external APIs.
    """
    name = "synthetic_ohlcv"

    def __init__(self, seed: int = 42):
        self.rng = np.random.default_rng(seed)

    def fetch_ohlcv(self, request: VendorRequest) -> pd.DataFrame:
        if request.interval != "1d":
            raise ValueError("SyntheticOHLCVVendor currently supports only daily interval '1d'.")

        dates = pd.date_range(
            start=request.start,
            end=request.end,
            freq="B",
            tz="UTC"
        )

        frames = []

        for symbol_index, symbol in enumerate(request.symbols):
            n = len(dates)

            base_price = 80 + 25 * symbol_index + self.rng.uniform(-5, 5)
            drift = self.rng.normal(loc=0.0002, scale=0.00005)
            volatility = self.rng.uniform(0.010, 0.025)

            log_returns = drift + volatility * self.rng.standard_normal(n)
            close = base_price * np.exp(np.cumsum(log_returns))

            overnight_noise = self.rng.normal(loc=0.0, scale=0.003, size=n)
            open_price = close * np.exp(overnight_noise)

            intraday_spread_high = np.abs(self.rng.normal(loc=0.004, scale=0.003, size=n))
            intraday_spread_low = np.abs(self.rng.normal(loc=0.004, scale=0.003, size=n))

            high = np.maximum(open_price, close) * (1.0 + intraday_spread_high)
            low = np.minimum(open_price, close) * (1.0 - intraday_spread_low)

            adjustment_factor = np.ones(n)
            if n > 250:
                adjustment_point = self.rng.integers(low=100, high=n)
                adjustment_factor[adjustment_point:] *= self.rng.choice([0.5, 0.75, 1.25])
            adj_close = close / adjustment_factor

            volume = self.rng.lognormal(mean=13.0, sigma=0.35, size=n).astype(float)

            frame = pd.DataFrame({
                "symbol": symbol,
                "timestamp": dates,
                "open": open_price,
                "high": high,
                "low": low,
                "close": close,
                "adj_close": adj_close,
                "volume": volume,
                "currency": request.currency,
                "source": self.name,
                "schema_version": "ohlcv_v1",
                "downloaded_at": pd.Timestamp.now(tz="UTC")
            })

            frames.append(frame)

        data = pd.concat(frames, ignore_index=True)
        data = data.sort_values(["symbol", "timestamp"]).reset_index(drop=True)

        return data

In [ ]:
vendor = SyntheticOHLCVVendor(seed=42)
raw_synthetic_data = vendor.fetch_ohlcv(request)

raw_synthetic_data.head()

## 7. Canonical column order and dtype targets

A stable column order makes downstream testing easier.

The dtype map below is not intended to replace validation. It simply documents the expected representation.

In [ ]:
CANONICAL_COLUMNS = [
    "symbol",
    "timestamp",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
    "currency",
    "source",
    "schema_version",
    "downloaded_at"
]

NUMERIC_COLUMNS = [
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume"
]

KEY_COLUMNS = [
    "symbol",
    "timestamp"
]

TEXT_COLUMNS = [
    "symbol",
    "currency",
    "source",
    "schema_version"
]

DATETIME_COLUMNS = [
    "timestamp",
    "downloaded_at"
]

## 8. Validation report objects

Validation should not only raise an exception.

For research infrastructure, it is useful to produce a structured report that can be logged, saved, or displayed in a notebook.

In [ ]:
@dataclass
class ValidationIssue:
    """
    One schema validation issue.
    """
    severity: str
    check: str
    message: str
    count: int | None = None


@dataclass
class ValidationReport:
    """
    Validation output for a market data frame.
    """
    is_valid: bool
    n_rows: int
    n_symbols: int
    n_issues: int
    issues: list[ValidationIssue]

    def to_frame(self) -> pd.DataFrame:
        if not self.issues:
            return pd.DataFrame(columns=["severity", "check", "message", "count"])

        return pd.DataFrame([asdict(issue) for issue in self.issues])

    def error_count(self) -> int:
        return sum(issue.severity == "error" for issue in self.issues)

    def warning_count(self) -> int:
        return sum(issue.severity == "warning" for issue in self.issues)

## 9. Standardisation helper

Different vendors often use different column names:

- `Date` vs `timestamp`;
- `Adj Close` vs `adj_close`;
- `Open` vs `open`;
- `Volume` vs `volume`.

The standardisation helper performs light column normalisation before validation.

In [ ]:
COLUMN_ALIASES = {
    "Date": "timestamp",
    "Datetime": "timestamp",
    "date": "timestamp",
    "datetime": "timestamp",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Adj Close": "adj_close",
    "Adj_Close": "adj_close",
    "adj close": "adj_close",
    "Volume": "volume",
    "Ticker": "symbol",
    "ticker": "symbol",
    "Symbol": "symbol"
}


def standardise_ohlcv_columns(
    df: pd.DataFrame,
    source: str,
    currency: str = "USD",
    schema_version: str = "ohlcv_v1"
) -> pd.DataFrame:
    """
    Rename common vendor columns and add missing metadata columns.
    """
    out = df.copy()

    out = out.rename(columns={col: COLUMN_ALIASES.get(col, col) for col in out.columns})

    if "adj_close" not in out.columns and "close" in out.columns:
        out["adj_close"] = out["close"]

    if "currency" not in out.columns:
        out["currency"] = currency

    if "source" not in out.columns:
        out["source"] = source

    if "schema_version" not in out.columns:
        out["schema_version"] = schema_version

    if "downloaded_at" not in out.columns:
        out["downloaded_at"] = pd.Timestamp.now(tz="UTC")

    canonical_present = [col for col in CANONICAL_COLUMNS if col in out.columns]
    extra_columns = [col for col in out.columns if col not in canonical_present]

    return out[canonical_present + extra_columns]

## 10. Schema validation function

The validator performs checks that are simple but high-value:

1. required columns exist;
2. timestamps are parseable and UTC-normalised;
3. numeric columns are numeric, finite, and economically sensible;
4. key columns are non-null;
5. duplicate `(symbol, timestamp)` rows are detected;
6. OHLC price relationships are respected;
7. data is sorted by symbol and timestamp;
8. schema version is consistent.

The function returns both a cleaned canonical DataFrame and a structured validation report.

In [ ]:
def validate_ohlcv_schema(
    df: pd.DataFrame,
    expected_schema_version: str = "ohlcv_v1",
    allow_extra_columns: bool = True
) -> tuple[pd.DataFrame, ValidationReport]:
    """
    Validate and lightly coerce a canonical OHLCV DataFrame.

    Returns
    -------
    cleaned:
        Cleaned DataFrame with canonical columns first.

    report:
        Structured validation report.
    """
    issues: list[ValidationIssue] = []
    cleaned = df.copy()

    missing_columns = [col for col in CANONICAL_COLUMNS if col not in cleaned.columns]

    if missing_columns:
        issues.append(ValidationIssue(
            severity="error",
            check="required_columns",
            message=f"Missing required columns: {missing_columns}",
            count=len(missing_columns)
        ))

        for col in missing_columns:
            cleaned[col] = pd.NA

    extra_columns = [col for col in cleaned.columns if col not in CANONICAL_COLUMNS]

    if extra_columns and not allow_extra_columns:
        issues.append(ValidationIssue(
            severity="error",
            check="extra_columns",
            message=f"Unexpected extra columns: {extra_columns}",
            count=len(extra_columns)
        ))

    cleaned = cleaned[CANONICAL_COLUMNS + extra_columns]

    for col in DATETIME_COLUMNS:
        try:
            cleaned[col] = pd.to_datetime(cleaned[col], utc=True)
        except Exception as exc:
            issues.append(ValidationIssue(
                severity="error",
                check=f"{col}_datetime_parse",
                message=f"Could not parse {col} as UTC datetime: {exc}",
                count=None
            ))

    for col in TEXT_COLUMNS:
        try:
            cleaned[col] = cleaned[col].astype("string")
        except Exception as exc:
            issues.append(ValidationIssue(
                severity="error",
                check=f"{col}_string_cast",
                message=f"Could not cast {col} to string dtype: {exc}",
                count=None
            ))

    for col in NUMERIC_COLUMNS:
        try:
            cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce").astype(float)
        except Exception as exc:
            issues.append(ValidationIssue(
                severity="error",
                check=f"{col}_numeric_cast",
                message=f"Could not cast {col} to float: {exc}",
                count=None
            ))

    for col in KEY_COLUMNS:
        null_count = int(cleaned[col].isna().sum())
        if null_count > 0:
            issues.append(ValidationIssue(
                severity="error",
                check=f"{col}_not_null",
                message=f"{col} contains null values.",
                count=null_count
            ))

    for col in NUMERIC_COLUMNS:
        null_count = int(cleaned[col].isna().sum())
        if null_count > 0:
            issues.append(ValidationIssue(
                severity="error",
                check=f"{col}_not_null",
                message=f"{col} contains missing or non-convertible values.",
                count=null_count
            ))

        values = cleaned[col].to_numpy(dtype=float, na_value=np.nan)
        non_finite_count = int((~np.isfinite(values)).sum())
        if non_finite_count > 0:
            issues.append(ValidationIssue(
                severity="error",
                check=f"{col}_finite",
                message=f"{col} contains non-finite values.",
                count=non_finite_count
            ))

    for col in ["open", "high", "low", "close", "adj_close"]:
        bad_count = int((cleaned[col] <= 0).sum())
        if bad_count > 0:
            issues.append(ValidationIssue(
                severity="error",
                check=f"{col}_positive",
                message=f"{col} must be positive.",
                count=bad_count
            ))

    bad_volume_count = int((cleaned["volume"] < 0).sum())
    if bad_volume_count > 0:
        issues.append(ValidationIssue(
            severity="error",
            check="volume_non_negative",
            message="volume must be non-negative.",
            count=bad_volume_count
        ))

    duplicate_count = int(cleaned.duplicated(KEY_COLUMNS).sum())
    if duplicate_count > 0:
        issues.append(ValidationIssue(
            severity="error",
            check="duplicate_symbol_timestamp",
            message="Duplicate rows found for composite key (symbol, timestamp).",
            count=duplicate_count
        ))

    high_low_bad = int((cleaned["high"] < cleaned["low"]).sum())
    if high_low_bad > 0:
        issues.append(ValidationIssue(
            severity="error",
            check="high_greater_equal_low",
            message="Some rows have high < low.",
            count=high_low_bad
        ))

    high_open_close_bad = int(
        (
            (cleaned["high"] < cleaned["open"]) |
            (cleaned["high"] < cleaned["close"])
        ).sum()
    )
    if high_open_close_bad > 0:
        issues.append(ValidationIssue(
            severity="error",
            check="high_bounds_open_close",
            message="Some rows have high below open or close.",
            count=high_open_close_bad
        ))

    low_open_close_bad = int(
        (
            (cleaned["low"] > cleaned["open"]) |
            (cleaned["low"] > cleaned["close"])
        ).sum()
    )
    if low_open_close_bad > 0:
        issues.append(ValidationIssue(
            severity="error",
            check="low_bounds_open_close",
            message="Some rows have low above open or close.",
            count=low_open_close_bad
        ))

    schema_mismatch_count = int((cleaned["schema_version"] != expected_schema_version).sum())
    if schema_mismatch_count > 0:
        issues.append(ValidationIssue(
            severity="error",
            check="schema_version",
            message=f"Rows do not match expected schema version {expected_schema_version}.",
            count=schema_mismatch_count
        ))

    not_sorted_symbols = []
    for symbol, group in cleaned.groupby("symbol", dropna=False):
        if not group["timestamp"].is_monotonic_increasing:
            not_sorted_symbols.append(str(symbol))

    if not_sorted_symbols:
        issues.append(ValidationIssue(
            severity="warning",
            check="timestamp_sorting",
            message=f"Timestamps are not sorted for symbols: {not_sorted_symbols[:10]}",
            count=len(not_sorted_symbols)
        ))

    gap_warnings = 0
    for symbol, group in cleaned.groupby("symbol", dropna=False):
        group = group.sort_values("timestamp")
        if len(group) < 3:
            continue

        observed_dates = pd.DatetimeIndex(group["timestamp"].dt.normalize().unique())
        expected_dates = pd.date_range(
            observed_dates.min(),
            observed_dates.max(),
            freq="B",
            tz="UTC"
        )

        missing_ratio = 1.0 - len(observed_dates.intersection(expected_dates)) / max(len(expected_dates), 1)

        if missing_ratio > 0.05:
            gap_warnings += 1

    if gap_warnings > 0:
        issues.append(ValidationIssue(
            severity="warning",
            check="business_day_gaps",
            message="Some symbols have more than 5% missing business-day observations.",
            count=gap_warnings
        ))

    cleaned = cleaned.sort_values(["symbol", "timestamp"]).reset_index(drop=True)

    n_rows = len(cleaned)
    n_symbols = int(cleaned["symbol"].nunique(dropna=True))
    error_count = sum(issue.severity == "error" for issue in issues)

    report = ValidationReport(
        is_valid=(error_count == 0),
        n_rows=n_rows,
        n_symbols=n_symbols,
        n_issues=len(issues),
        issues=issues
    )

    return cleaned, report

In [ ]:
canonical_data = standardise_ohlcv_columns(
    raw_synthetic_data,
    source=vendor.name,
    currency=request.currency
)

validated_data, validation_report = validate_ohlcv_schema(canonical_data)

print("Is valid:", validation_report.is_valid)
print("Rows:", validation_report.n_rows)
print("Symbols:", validation_report.n_symbols)
print("Issues:", validation_report.n_issues)

validation_report.to_frame()

## 11. Visual sanity check

Schema validation catches many mechanical issues, but visual checks are still useful.

A sudden price level shift, a flatlined asset, or a suspicious volume pattern may be technically valid but still economically suspicious.

In [ ]:
sample_symbol = request.symbols[0]

plot_data = validated_data[validated_data["symbol"] == sample_symbol].copy()

plt.figure(figsize=(10, 6))
plt.plot(plot_data["timestamp"], plot_data["close"], label="close")
plt.plot(plot_data["timestamp"], plot_data["adj_close"], label="adj_close", alpha=0.75)
plt.title(f"Synthetic close and adjusted close: {sample_symbol}")
plt.xlabel("Timestamp")
plt.ylabel("Price")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(plot_data["timestamp"], plot_data["volume"])
plt.title(f"Synthetic volume: {sample_symbol}")
plt.xlabel("Timestamp")
plt.ylabel("Volume")
plt.show()

## 12. Failure demonstration

A good validator should fail loudly when the data contract is broken.

Below, we intentionally corrupt a small copy of the data:

1. duplicate a row;
2. make one price negative;
3. make one `high` less than `low`;
4. remove a required value.

The report should catch these problems.

In [ ]:
broken_data = validated_data.copy()

broken_data = pd.concat([broken_data, broken_data.iloc[[0]]], ignore_index=True)

broken_data.loc[10, "close"] = -1.0
broken_data.loc[20, "high"] = broken_data.loc[20, "low"] - 0.01
broken_data.loc[30, "symbol"] = pd.NA

_, broken_report = validate_ohlcv_schema(broken_data)

print("Is valid:", broken_report.is_valid)
broken_report.to_frame()

## 13. Dataset fingerprinting

A dataset fingerprint is a compact way of identifying the exact contents of a dataset.

It is useful for:

- cache invalidation;
- experiment tracking;
- debugging;
- reproducibility;
- detecting whether a vendor silently changed historical data.

The fingerprint below hashes a stable CSV representation of the canonical data.

For large production datasets, one would usually hash files, partitions, or metadata rather than converting the whole DataFrame to CSV.

In [ ]:
def dataframe_fingerprint(df: pd.DataFrame, columns: list[str] | None = None) -> str:
    """
    Compute a stable SHA-256 fingerprint for a DataFrame.

    This is suitable for small research datasets.
    For larger datasets, prefer file-level or partition-level hashing.
    """
    if columns is None:
        columns = list(df.columns)

    stable = df[columns].copy()
    stable = stable.sort_values(KEY_COLUMNS).reset_index(drop=True)

    csv_bytes = stable.to_csv(index=False).encode("utf-8")
    return hashlib.sha256(csv_bytes).hexdigest()


fingerprint = dataframe_fingerprint(validated_data, columns=CANONICAL_COLUMNS)

fingerprint

## 14. Manifest generation

A manifest records metadata about the dataset.

A useful manifest should answer:

- which symbols were fetched?
- what date range was requested?
- which source generated the data?
- which schema version was used?
- how many rows were produced?
- did validation pass?
- what is the dataset fingerprint?

In [ ]:
def build_dataset_manifest(
    data: pd.DataFrame,
    request: VendorRequest,
    validation_report: ValidationReport,
    fingerprint: str
) -> dict:
    """
    Build a JSON-serialisable dataset manifest.
    """
    timestamp_min = data["timestamp"].min()
    timestamp_max = data["timestamp"].max()

    manifest = {
        "dataset_name": "canonical_ohlcv",
        "schema_version": "ohlcv_v1",
        "source": sorted(data["source"].dropna().astype(str).unique().tolist()),
        "requested_symbols": request.symbols,
        "observed_symbols": sorted(data["symbol"].dropna().astype(str).unique().tolist()),
        "requested_start": request.start,
        "requested_end": request.end,
        "observed_start": None if pd.isna(timestamp_min) else timestamp_min.isoformat(),
        "observed_end": None if pd.isna(timestamp_max) else timestamp_max.isoformat(),
        "interval": request.interval,
        "currency": request.currency,
        "row_count": int(len(data)),
        "validation_passed": validation_report.is_valid,
        "validation_issue_count": validation_report.n_issues,
        "fingerprint_sha256": fingerprint,
        "created_at": pd.Timestamp.now(tz="UTC").isoformat()
    }

    return manifest


manifest = build_dataset_manifest(
    data=validated_data,
    request=request,
    validation_report=validation_report,
    fingerprint=fingerprint
)

manifest

## 15. Saving validated data

For research-scale data, Parquet is usually preferable to CSV because it preserves column types better and supports efficient columnar reads.

This notebook attempts to save as Parquet. If no Parquet engine is installed, it falls back to CSV.

The manifest is saved as JSON.

In [ ]:
output_dir = Path("data/processed/ohlcv_v1")
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / "synthetic_ohlcv.parquet"
csv_path = output_dir / "synthetic_ohlcv.csv"
manifest_path = output_dir / "manifest.json"

try:
    validated_data.to_parquet(parquet_path, index=False)
    saved_data_path = parquet_path
    print(f"Saved Parquet file to: {parquet_path}")
except Exception as exc:
    warnings.warn(f"Could not save Parquet file; falling back to CSV. Reason: {exc}")
    validated_data.to_csv(csv_path, index=False)
    saved_data_path = csv_path
    print(f"Saved CSV file to: {csv_path}")

manifest_path.write_text(json.dumps(manifest, indent=2))

print(f"Saved manifest to: {manifest_path}")

## 16. Loading and revalidating saved data

A useful persistence test is:

1. save the validated dataset;
2. reload it;
3. revalidate it;
4. check that the fingerprint is stable.

This catches type-loss issues introduced by the storage format.

In [ ]:
def load_saved_dataset(path: Path) -> pd.DataFrame:
    """
    Load a saved dataset from Parquet or CSV.
    """
    if path.suffix == ".parquet":
        return pd.read_parquet(path)

    if path.suffix == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Unsupported file type: {path.suffix}")


reloaded_data_raw = load_saved_dataset(saved_data_path)
reloaded_data = standardise_ohlcv_columns(
    reloaded_data_raw,
    source=vendor.name,
    currency=request.currency
)

revalidated_data, revalidation_report = validate_ohlcv_schema(reloaded_data)

reloaded_fingerprint = dataframe_fingerprint(revalidated_data, columns=CANONICAL_COLUMNS)

print("Revalidation passed:", revalidation_report.is_valid)
print("Fingerprint stable:", fingerprint == reloaded_fingerprint)

## 17. Optional adapter: Yahoo Finance through yfinance

The repository is designed to use public data where possible.

The adapter below shows how a Yahoo Finance / `yfinance` vendor could fit into the same interface.

This cell is optional because it requires:

1. the `yfinance` package;
2. internet access;
3. a compatible runtime environment.

The key point is not that Yahoo Finance is the final institutional data source. The key point is that every vendor should be normalised into the same canonical schema before downstream models consume it.

In [ ]:
class YFinanceVendor:
    """
    Optional Yahoo Finance adapter using yfinance.

    This adapter is intentionally lightweight and research-oriented.
    It should be wrapped with caching, retry logic, and stronger audit controls
    before being used in a larger research system.
    """
    name = "yfinance"

    def fetch_ohlcv(self, request: VendorRequest) -> pd.DataFrame:
        try:
            import yfinance as yf
        except ImportError as exc:
            raise ImportError(
                "yfinance is not installed. Install it with: pip install yfinance"
            ) from exc

        raw = yf.download(
            tickers=request.symbols,
            start=request.start,
            end=request.end,
            interval=request.interval,
            auto_adjust=request.auto_adjust,
            group_by="ticker",
            progress=False,
            threads=True
        )

        frames = []

        if isinstance(raw.columns, pd.MultiIndex):
            for symbol in request.symbols:
                if symbol not in raw.columns.get_level_values(0):
                    continue

                part = raw[symbol].reset_index()
                part["symbol"] = symbol
                frames.append(part)
        else:
            symbol = request.symbols[0]
            part = raw.reset_index()
            part["symbol"] = symbol
            frames.append(part)

        if not frames:
            raise ValueError("No data returned by yfinance.")

        data = pd.concat(frames, ignore_index=True)

        data = standardise_ohlcv_columns(
            data,
            source=self.name,
            currency=request.currency,
            schema_version="ohlcv_v1"
        )

        return data


# Example usage if internet and yfinance are available:
#
# yf_request = VendorRequest(
#     symbols=["SPY", "QQQ"],
#     start="2022-01-01",
#     end="2024-12-31",
#     interval="1d",
#     currency="USD",
#     auto_adjust=False
# )
#
# yf_vendor = YFinanceVendor()
# yf_data = yf_vendor.fetch_ohlcv(yf_request)
# yf_validated, yf_report = validate_ohlcv_schema(yf_data)
# yf_report.to_frame()

## 18. Data vendor abstraction test

Even if vendors differ internally, they should expose the same method.

This makes it easier for future notebooks to swap between:

- synthetic data;
- Yahoo Finance;
- CSV files;
- local Parquet;
- paid vendor data;
- database-backed data.

The downstream model should not care where the data came from, as long as the schema contract is satisfied.

In [ ]:
def fetch_validate_and_manifest(
    vendor: MarketDataVendor,
    request: VendorRequest
) -> tuple[pd.DataFrame, ValidationReport, dict]:
    """
    End-to-end fetch, standardise, validate, and manifest workflow.
    """
    raw_data = vendor.fetch_ohlcv(request)

    canonical = standardise_ohlcv_columns(
        raw_data,
        source=vendor.name,
        currency=request.currency,
        schema_version="ohlcv_v1"
    )

    validated, report = validate_ohlcv_schema(canonical)

    dataset_hash = dataframe_fingerprint(validated, columns=CANONICAL_COLUMNS)

    manifest = build_dataset_manifest(
        data=validated,
        request=request,
        validation_report=report,
        fingerprint=dataset_hash
    )

    return validated, report, manifest


test_data, test_report, test_manifest = fetch_validate_and_manifest(vendor, request)

print("Valid:", test_report.is_valid)
print("Rows:", len(test_data))
print("Fingerprint:", test_manifest["fingerprint_sha256"][:16] + "...")

## 19. Schema evolution

Data schemas evolve.

Examples:

- adding `exchange`;
- adding `asset_type`;
- adding `vendor_symbol`;
- adding `corporate_action_flag`;
- adding `contract_month` for futures;
- adding `bid`, `ask`, and `mid` for quote data.

A serious repo should version schemas rather than silently changing them.

For example:

| Schema version | Purpose |
|---|---|
| `ohlcv_v1` | Daily OHLCV bars |
| `ohlcv_v2` | Adds `exchange`, `asset_type`, and `vendor_symbol` |
| `futures_ohlcv_v1` | Adds `contract`, `expiry`, and `roll_rule` |
| `quotes_l1_v1` | Adds `bid`, `ask`, `mid`, and spread fields |
| `lob_l2_v1` | Adds depth-level order book data |

This notebook only implements `ohlcv_v1`, but it establishes the versioning pattern.

## 20. Practical data quality checklist

Before a model uses market data, check:

1. **Schema**  
   Are all required columns present?

2. **Dtypes**  
   Are timestamps, symbols, and numeric values represented consistently?

3. **Timezone**  
   Are timestamps timezone-aware and normalised?

4. **Composite key uniqueness**  
   Is `(symbol, timestamp)` unique?

5. **OHLC logic**  
   Do high/low/open/close relationships make sense?

6. **Missingness**  
   Are gaps expected or suspicious?

7. **Adjustment policy**  
   Are adjusted and unadjusted prices clearly separated?

8. **Point-in-time safety**  
   Could any field contain information unavailable at the timestamp?

9. **Vendor provenance**  
   Can the data source be traced?

10. **Reproducibility**  
   Can the same dataset be regenerated or fingerprinted?

## 21. Limitations

This notebook deliberately implements a lightweight schema system.

### 21.1 Synthetic data is not market data

The synthetic vendor produces clean, controlled OHLCV data.

Real vendor data can include:

- missing sessions;
- duplicate rows;
- stock splits;
- dividends;
- symbol changes;
- delistings;
- timezone inconsistencies;
- stale prices;
- vendor revisions.

### 21.2 Daily bars hide microstructure problems

Daily OHLCV data is much cleaner than tick, quote, or order-book data.

High-frequency data introduces additional issues:

- out-of-order events;
- exchange sequence numbers;
- crossed markets;
- locked markets;
- auction prints;
- trade corrections;
- quote stuffing;
- latency;
- daylight saving time.

### 21.3 Validation rules are domain-specific

A rule that is correct for equities may not be correct for futures, FX, rates, crypto, or options.

For example:

- futures need contract expiry and roll metadata;
- FX volume may be missing or proxy-based;
- options need strike, expiry, right, implied volatility, and Greeks;
- rates data can have negative values.

### 21.4 Vendor data is not automatically point-in-time

A dataset may accidentally contain revised or adjusted information that was not known at the historical timestamp.

This is especially dangerous in fundamental, macro, and corporate-action data.

### 21.5 Manual validation does not replace a full data quality system

The validator here is intentionally transparent and easy to read.

A larger system would use formal data contracts, automated validation suites, alerts, lineage tracking, and persistent metadata.

## 22. Research frontier and current directions

Modern quant data infrastructure is moving beyond simple CSV downloads and ad hoc cleaning scripts.

### 22.1 Data contracts for research pipelines

A data contract defines what a dataset must look like before another system is allowed to consume it.

In quant research, contracts are useful because they prevent models from silently adapting to broken data.

A future notebook could define formal contracts for OHLCV, futures curves, options chains, and order-book snapshots.

### 22.2 Point-in-time and bitemporal data

Point-in-time data asks:

> What did we know at the time?

Bitemporal data tracks both:

- the time the event happened;
- the time the information became available or was revised.

This matters for avoiding look-ahead bias in fundamental, macro, and corporate-action datasets.

A future notebook could simulate a revised macroeconomic time series and show how using final revised values creates look-ahead bias.

### 22.3 Columnar formats and lakehouse storage

Modern analytics systems increasingly use columnar formats such as Apache Arrow and Parquet.

These formats are useful because financial research often reads only a subset of columns over a large time range.

A future notebook could compare CSV, Parquet, and Arrow-backed workflows for a large synthetic OHLCV dataset.

### 22.4 Data quality observability

A production data system should not merely validate once.

It should monitor:

- missingness rates;
- row counts;
- duplicate keys;
- distribution shifts;
- stale symbols;
- vendor outages;
- unusual price jumps;
- schema changes.

A future notebook could build a simple data quality dashboard that tracks validation metrics over time.

### 22.5 Vendor reconciliation

Institutional data teams often compare multiple vendors.

If two sources disagree on a price, volume, corporate action, or timestamp, the system should flag the discrepancy instead of blindly trusting one source.

A future notebook could compare two synthetic vendors with controlled disagreements and design a reconciliation report.

### 22.6 Streaming data contracts

For intraday and live systems, validation cannot wait until the end of the day.

Streaming contracts validate events as they arrive.

This is especially important for order-book data, where one bad event can corrupt the reconstructed book state.

A future notebook could define an event-level schema for trades and quotes, then validate a synthetic stream online.

## 23. Suggested follow-up notebooks

This notebook naturally leads to:

1. **01_02_brownian_bridge_imputation**  
   Handling missing observations once the canonical schema is established.

2. **01_03_continuous_futures_contract_rolling**  
   Extending the schema to futures contracts, expiries, and roll metadata.

3. **01_04_return_sanitization_and_bias_control**  
   Preventing survivorship bias, adjustment confusion, and leakage.

4. **01_05_empirical_distribution_analyzer**  
   Studying return distributions after data validation.

5. **01_07_futures_calendar_and_roll_yield_decomposition**  
   Moving from spot-like price series to futures curve data.

6. **05_01_vectorized_backtest_engine**  
   Using validated canonical data as the input to a backtest engine.

7. **07_01_multi_asset_cta_research_pipeline**  
   Combining data ingestion, validation, signal generation, portfolio construction, and backtesting.

## 24. Summary

This notebook built the first data infrastructure layer of the repository.

The main components were:

1. a vendor-neutral request object;
2. a vendor interface;
3. a synthetic offline OHLCV vendor;
4. a canonical OHLCV schema;
5. a schema standardisation helper;
6. a validation report system;
7. a dataset fingerprint;
8. a JSON manifest;
9. an optional `yfinance` adapter;
10. a schema evolution and data quality checklist.

The key computational takeaway is:

> A quant research system should not allow arbitrary vendor output to flow directly into models. Data should first pass through a stable interface, canonical schema, and validation layer.

The key financial takeaway is:

> Bad data can create fake alpha. A rigorous data contract is a defence against silent modelling errors, look-ahead bias, and vendor-specific quirks.

## 25. Further reading

### Data validation and contracts

- Pandera documentation.
- Great Expectations documentation.
- Pydantic documentation.
- Delta Lake schema enforcement and schema evolution documentation.

### Columnar analytics and storage

- Apache Arrow documentation.
- Apache Parquet documentation.
- DuckDB Parquet documentation.
- pandas PyArrow backend documentation.

### Financial data engineering

- Point-in-time data and bitemporal modelling.
- Survivorship bias in equity datasets.
- Corporate action adjustment methodology.
- Futures contract rolling and curve construction.
- Market microstructure event validation.